# 06 — Grafici interattivi Altair

Mappa del mondo statica e con slider temporale, ed heatmap interattiva "chi domina ogni sport" — tutte in Altair/Vega-Lite (grafici HTML interattivi, richiedono connessione internet per lo sfondo geografico `world-atlas`).

**File necessari:**
- `olympics_with_clusters.csv` (da `02_Clustering_Metodologia.ipynb`)
- `olympics_dataset.csv` (Kaggle, medaglie)
- `olympics_sport_cluster.csv`, `mobility_dynamic.csv` (da `03_Cluster_Dinamico.ipynb`)

In [ ]:
!pip install altair vega_datasets

In [ ]:
# -----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import altair as alt
from vega_datasets import data as vega_data

# Altair: disabilita il limite di 5000 righe per dataset grandi
alt.data_transformers.disable_max_rows()

CLUSTER_ORDER  = ['Ricchi avanzati', 'Reddito medio-alto',
                  'Reddito basso-medio', 'Poveri estremi']
CLUSTER_COLORS = ['#185FA5', '#3B6D11', '#BA7517', '#A32D2D']



In [ ]:
IOC_TO_ISO3 = {
    'ALG':'DZA','ANG':'AGO','ARG':'ARG','ARM':'ARM','ARU':'ABW','ASA':'ASM',
    'AUS':'AUS','AUT':'AUT','AZE':'AZE','BAH':'BHS','BAN':'BGD','BAR':'BRB',
    'BDI':'BDI','BEL':'BEL','BEN':'BEN','BER':'BMU','BHU':'BTN','BIH':'BIH',
    'BIZ':'BLZ','BLR':'BLR','BOL':'BOL','BOT':'BWA','BRA':'BRA','BRN':'BHR',
    'BUL':'BGR','BUR':'BFA','CAF':'CAF','CAM':'KHM','CAN':'CAN','CAY':'CYM',
    'CGO':'COG','CHA':'TCD','CHI':'CHL','CHN':'CHN','CIV':'CIV','CMR':'CMR',
    'COD':'COD','COL':'COL','COM':'COM','CPV':'CPV','CRC':'CRI','CRO':'HRV',
    'CUB':'CUB','CYP':'CYP','CZE':'CZE','DEN':'DNK','DJI':'DJI','DOM':'DOM',
    'ECU':'ECU','EGY':'EGY','ESA':'SLV','ESP':'ESP','EST':'EST','ETH':'ETH',
    'FIJ':'FJI','FIN':'FIN','FRA':'FRA','FSM':'FSM','GAB':'GAB','GAM':'GMB',
    'GBS':'GNB','GBR':'GBR','GEO':'GEO','GEQ':'GNQ','GER':'DEU','GHA':'GHA',
    'GRE':'GRC','GRN':'GRD','GUA':'GTM','GUI':'GIN','GUM':'GUM','GUY':'GUY',
    'HAI':'HTI','HON':'HND','HKG':'HKG','HUN':'HUN','INA':'IDN','IND':'IND',
    'IRI':'IRN','IRL':'IRL','IRQ':'IRQ','ISL':'ISL','ISR':'ISR','ISV':'VIR',
    'ITA':'ITA','IVB':'VGB','JAM':'JAM','JOR':'JOR','JPN':'JPN','KAZ':'KAZ',
    'KEN':'KEN','KGZ':'KGZ','KIR':'KIR','KOR':'KOR','KSA':'SAU','KUW':'KWT',
    'LAO':'LAO','LAT':'LVA','LBA':'LBY','LBN':'LBN','LBR':'LBR','LES':'LSO',
    'LIE':'LIE','LTU':'LTU','LUX':'LUX','MAD':'MDG','MAR':'MAR','MAS':'MYS',
    'MAW':'MWI','MDA':'MDA','MDV':'MDV','MEX':'MEX','MGL':'MNG','MKD':'MKD',
    'MLI':'MLI','MLT':'MLT','MON':'MCO','MOZ':'MOZ','MRI':'MUS','MTN':'MRT',
    'MYA':'MMR','NAM':'NAM','NCA':'NIC','NED':'NLD','NEP':'NPL','NGR':'NGA',
    'NIG':'NER','NOR':'NOR','NRU':'NRU','NZL':'NZL','OMA':'OMN','PAK':'PAK',
    'PAN':'PAN','PAR':'PRY','PER':'PER','PHI':'PHL','PLE':'PSE','PLW':'PLW',
    'PNG':'PNG','POL':'POL','POR':'PRT','PRK':'PRK','PUR':'PRI','QAT':'QAT',
    'ROU':'ROU','RSA':'ZAF','RUS':'RUS','RWA':'RWA','SAM':'WSM','SEN':'SEN',
    'SEY':'SYC','SGP':'SGP','SKN':'KNA','SLE':'SLE','SLO':'SVN','SMR':'SMR',
    'SOL':'SLB','SOM':'SOM','SRI':'LKA','SSD':'SSD','STP':'STP','SUD':'SDN',
    'SUI':'CHE','SUR':'SUR','SVK':'SVK','SWE':'SWE','SWZ':'SWZ','SYR':'SYR',
    'TAN':'TZA','TGA':'TON','THA':'THA','TJK':'TJK','TKM':'TKM','TLS':'TLS',
    'TOG':'TGO','TPE':'TWN','TTO':'TTO','TUN':'TUN','TUR':'TUR','TUV':'TUV',
    'UAE':'ARE','UGA':'UGA','UKR':'UKR','URU':'URY','USA':'USA','UZB':'UZB',
    'VAN':'VUT','VEN':'VEN','VIE':'VNM','VIN':'VCT','YEM':'YEM','ZAM':'ZMB',
    'ZIM':'ZWE',
    # Paesi storici
    'URS':'RUS','TCH':'CZE','YUG':'SRB','GDR':'DEU','FRG':'DEU','SCG':'SRB',
}

In [ ]:
ISO3_TO_NUMERIC = {
    'AFG':4,'ALB':8,'DZA':12,'AND':20,'AGO':24,'ATG':28,'ARG':32,'ARM':51,
    'AUS':36,'AUT':40,'AZE':31,'BHS':44,'BHR':48,'BGD':50,'BRB':52,'BLR':112,
    'BEL':56,'BLZ':84,'BEN':204,'BTN':64,'BOL':68,'BIH':70,'BWA':72,'BRA':76,
    'BRN':96,'BGR':100,'BFA':854,'BDI':108,'CPV':132,'KHM':116,'CMR':120,
    'CAN':124,'CAF':140,'TCD':148,'CHL':152,'CHN':156,'COL':170,'COM':174,
    'COD':180,'COG':178,'CRI':188,'CIV':384,'HRV':191,'CUB':192,'CYP':196,
    'CZE':203,'DNK':208,'DJI':262,'DOM':214,'ECU':218,'EGY':818,'SLV':222,
    'GNQ':226,'ERI':232,'EST':233,'SWZ':748,'ETH':231,'FJI':242,'FIN':246,
    'FRA':250,'GAB':266,'GMB':270,'GEO':268,'DEU':276,'GHA':288,'GRC':300,
    'GRD':308,'GTM':320,'GIN':324,'GNB':624,'GUY':328,'HTI':332,'HND':340,
    'HUN':348,'ISL':352,'IND':356,'IDN':360,'IRN':364,'IRQ':368,'IRL':372,
    'ISR':376,'ITA':380,'JAM':388,'JPN':392,'JOR':400,'KAZ':398,'KEN':404,
    'KIR':296,'PRK':408,'KOR':410,'KWT':414,'KGZ':417,'LAO':418,'LVA':428,
    'LBN':422,'LSO':426,'LBR':430,'LBY':434,'LIE':438,'LTU':440,'LUX':442,
    'MDG':450,'MWI':454,'MYS':458,'MDV':462,'MLI':466,'MLT':470,'MRT':478,
    'MUS':480,'MEX':484,'FSM':583,'MDA':498,'MCO':492,'MNG':496,'MNE':499,
    'MAR':504,'MOZ':508,'MMR':104,'NAM':516,'NRU':520,'NPL':524,'NLD':528,
    'NZL':554,'NIC':558,'NER':562,'NGA':566,'NOR':578,'OMN':512,'PAK':586,
    'PLW':585,'PSE':275,'PAN':591,'PNG':598,'PRY':600,'PER':604,'PHL':608,
    'POL':616,'PRT':620,'QAT':634,'ROU':642,'RUS':643,'RWA':646,'KNA':659,
    'VCT':670,'WSM':882,'SMR':674,'STP':678,'SAU':682,'SEN':686,'SRB':688,
    'SYC':690,'SLE':694,'SGP':702,'SVK':703,'SVN':705,'SLB':90,'SOM':706,
    'ZAF':710,'SSD':728,'ESP':724,'LKA':144,'SDN':736,'SUR':740,'SWE':752,
    'CHE':756,'SYR':760,'TWN':158,'TJK':762,'TZA':834,'THA':764,'TLS':626,
    'TGO':768,'TON':776,'TTO':780,'TUN':788,'TUR':792,'TKM':795,'TUV':798,
    'UGA':800,'UKR':804,'ARE':784,'GBR':826,'USA':840,'URY':858,'UZB':860,
    'VUT':548,'VEN':862,'VNM':704,'YEM':887,'ZMB':894,'ZWE':716,
    'PRI':630,'HKG':344,'MAC':446,'GUM':316,'VIR':850,'ABW':533,
}


In [ ]:
clustered = pd.read_csv('olympics_with_clusters.csv')

In [ ]:
# FIX: gestisce NOC in formato IOC e in formato ISO3 (misti nel dataset)
all_clustered = clustered[clustered['cluster_wb'].notna()].copy()

# Step 1: prova mapping IOC -> ISO3
all_clustered['iso3'] = all_clustered['noc'].map(IOC_TO_ISO3)

# Step 2: per i NOC non trovati nel mapping, controlla se sono già ISO3
mask_missing = all_clustered['iso3'].isna()
all_clustered.loc[mask_missing, 'iso3'] = all_clustered.loc[mask_missing, 'noc'].apply(
    lambda x: x if x in ISO3_TO_NUMERIC else None
)

# Step 3: mappa a numeric_id e deduplica tenendo il record più recente per paese
all_clustered['numeric_id'] = all_clustered['iso3'].map(ISO3_TO_NUMERIC)

map_data = (
    all_clustered[all_clustered['numeric_id'].notna()]
    .sort_values('year', ascending=False)
    .drop_duplicates(subset='numeric_id', keep='first')
    [['iso3', 'numeric_id', 'country', 'cluster_wb',
      'gdp_per_capita', 'life_expectancy', 'urbanization_pct', 'infant_mortality']]
    .copy()
)
map_data['numeric_id'] = map_data['numeric_id'].apply(lambda x: f'{int(x):03d}')

print(f"Paesi mappabili: {len(map_data)}")
print(map_data['cluster_wb'].value_counts().to_string())


In [ ]:
# %% [3] MAPPA DEL MONDO — CLUSTER SOCIOECONOMICI
# -----------------------------------------------------------------------------
WORLD_TOPO_URL = "https://cdn.jsdelivr.net/npm/world-atlas@2/countries-110m.json"
world_geo = alt.topo_feature(WORLD_TOPO_URL, 'countries')

background = alt.Chart(world_geo).mark_geoshape(
    fill='#e0e0e0', stroke='white', strokeWidth=0.4
).project('naturalEarth1')

choropleth = alt.Chart(world_geo).mark_geoshape(
    stroke='white', strokeWidth=0.4
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(
        data=map_data,
        key='numeric_id',
        fields=['country', 'cluster_wb', 'gdp_per_capita',
                'life_expectancy', 'urbanization_pct']
    )
).transform_filter(
    'datum.cluster_wb != null'
).encode(
    color=alt.Color(
        'cluster_wb:N',
        scale=alt.Scale(domain=CLUSTER_ORDER, range=CLUSTER_COLORS),
        legend=alt.Legend(
            title='Cluster socioeconomico',
            orient='bottom',
            titleFontSize=11,
            labelFontSize=10,
            columns=2,
        )
    ),
    tooltip=[
        alt.Tooltip('country:N',          title='Paese'),
        alt.Tooltip('cluster_label:N',    title='Cluster'),
        alt.Tooltip('gdp_per_capita:Q',   title='PIL pro capite (USD)', format=',.0f'),
        alt.Tooltip('life_expectancy:Q',  title='Aspett. vita', format='.1f'),
        alt.Tooltip('urbanization_pct:Q', title='Urbaniz. %', format='.1f'),
    ]
).project('naturalEarth1')

map_cluster = (background + choropleth).properties(
    width=750, height=420,
    title=alt.TitleParams(
        text='Cluster socioeconomici dei paesi olimpici',
        subtitle='Basato su PIL pro capite, aspettativa di vita, urbanizzazione, mortalità infantile',
        fontSize=14, subtitleFontSize=11
    )
).configure_view(stroke=None)

map_cluster.save('mappa_cluster.html')
print("✓ Salvato: mappa_cluster.html")
map_cluster

In [ ]:
sport_df = pd.read_csv('olympics_sport_cluster.csv')

event_df = (
    sport_df
    .groupby(['Year', 'NOC', 'Sport', 'Event', 'Medal', 'cluster_label'])
    .size().reset_index(name='n')
)
event_df['count'] = 1

sport_cluster = (
    event_df
    .groupby(['cluster_label', 'Sport'])['count']
    .sum().reset_index()
)
totals = sport_cluster.groupby('cluster_label')['count'].sum().reset_index()
totals.columns = ['cluster_label', 'total']
sport_cluster = sport_cluster.merge(totals, on='cluster_label')
sport_cluster['pct'] = (sport_cluster['count'] / sport_cluster['total'] * 100).round(2)

## MAPPA DEL MONDO — LINEA TEMPORALE INTERATTIVA

La mappa sopra è un'istantanea statica (cluster più recente disponibile per
paese). Qui costruiamo una versione con uno **slider temporale**: scorrendo
tra le sole edizioni olimpiche reali, ogni paese prende il colore del
cluster socioeconomico a cui apparteneva **in quell'anno specifico**
(cluster dinamico, stesso modello multivariato usato in tutto il resto del
notebook — `mobility`, calcolata più sopra nella sezione "ANALISI SPORT").
Il tooltip mostra anche il numero di medaglie vinte dal paese in quella
edizione.

In [ ]:
# %% [3bis] PREPARAZIONE DATI — CLUSTER DINAMICO PER EDIZIONE OLIMPICA
# -----------------------------------------------------------------------------
# 'mobility' qui è caricata da 'mobility_dynamic.csv' (prodotta da
# 03_Cluster_Dinamico.ipynb), invece di essere ricalcolata da zero: stessa
# tabella (noc_ioc, country, year, cluster_dynamic) usata in quel notebook.

kaggle = pd.read_csv('olympics_dataset.csv')
mobility = pd.read_csv('mobility_dynamic.csv')

medals_by_country_year = (
    event_df.groupby(['Year', 'NOC'])['count'].sum().reset_index(name='medaglie')
)

time_base = mobility[['noc_ioc', 'country', 'year', 'cluster_dynamic']].drop_duplicates()
time_base = time_base.merge(
    medals_by_country_year,
    left_on=['noc_ioc', 'year'], right_on=['NOC', 'Year'], how='left'
).drop(columns=['NOC', 'Year'])
time_base['medaglie'] = time_base['medaglie'].fillna(0).astype(int)

# Teniamo solo le edizioni olimpiche REALI (gli anni presenti nel dataset
# Kaggle), non ogni singolo anno del dataset socioeconomico: gli anni non
# sono equispaziati (ogni 4 anni fino al 1992, poi ogni 2 alternando
# Olimpiadi estive e invernali), quindi usiamo un indice progressivo
# (0..N-1) per uno slider che scorre solo tra edizioni reali.
olympic_years = sorted(kaggle['Year'].unique())
year_to_idx = {y: i for i, y in enumerate(olympic_years)}

time_base = time_base[time_base['year'].isin(olympic_years)].copy()
time_base['year_idx'] = time_base['year'].map(year_to_idx)

# ISO3 + numeric_id per il join con la mappa (stessa logica della mappa statica)
time_base['iso3'] = time_base['noc_ioc'].map(IOC_TO_ISO3)
mask_missing = time_base['iso3'].isna()
time_base.loc[mask_missing, 'iso3'] = time_base.loc[mask_missing, 'noc_ioc'].apply(
    lambda x: x if x in ISO3_TO_NUMERIC else None
)
time_base['numeric_id'] = time_base['iso3'].map(ISO3_TO_NUMERIC)
time_base = time_base[time_base['numeric_id'].notna()].copy()
time_base['numeric_id'] = time_base['numeric_id'].apply(lambda x: f'{int(x):03d}')

print(f"Edizioni olimpiche incluse: {len(olympic_years)} (dal {olympic_years[0]} al {olympic_years[-1]})")
print(f"Righe mappa temporale (paese × edizione): {len(time_base):,}")

In [ ]:
# %% [3ter] MAPPA DEL MONDO CON SLIDER — CLUSTER DINAMICO PER EDIZIONE
# -----------------------------------------------------------------------------
# Trucco tecnico: filtriamo PRIMA la tabella paese-edizione (time_base) in
# base allo slider, e SOLO DOPO facciamo il lookup della geometria da
# world_geo (direzione inversa rispetto alla mappa statica sopra). Se invece
# si facesse il lookup prima di filtrare, ogni paese avrebbe più righe
# candidate (una per edizione) e Vega-Lite ne userebbe una arbitraria e
# fissa, rendendo la mappa NON reattiva allo slider.

year_slider = alt.binding_range(
    min=15, max=len(olympic_years) - 1, step=1,
    name='Edizione olimpica (indice): '
)
year_select = alt.selection_point(
    name='year_sel',
    fields=['year_idx'],
    bind=year_slider,
    value=[{'year_idx': len(olympic_years) - 1}]  # parte dall'edizione più recente
)

choropleth_time = alt.Chart(time_base).transform_filter(
    year_select
).transform_lookup(
    lookup='numeric_id',
    from_=alt.LookupData(data=world_geo, key='id', fields=['type', 'geometry'])
).mark_geoshape(
    stroke='white', strokeWidth=0.4
).encode(
    color=alt.Color(
        'cluster_dynamic:N',
        scale=alt.Scale(domain=CLUSTER_ORDER, range=CLUSTER_COLORS),
        legend=alt.Legend(
            title="Cluster socioeconomico (di quell'anno)",
            orient='bottom', titleFontSize=11, labelFontSize=10, columns=2,
        )
    ),
    tooltip=[
        alt.Tooltip('country:N',         title='Paese'),
        alt.Tooltip('year:O',            title='Anno'),
        alt.Tooltip('cluster_dynamic:N', title='Cluster in quel anno'),
        alt.Tooltip('medaglie:Q',        title='Medaglie vinte in quel anno'),
    ]
).add_params(
    year_select
).project('naturalEarth1')

# Annotazione con l'anno effettivamente selezionato (lo slider mostra solo
# l'indice dell'edizione, non l'anno per esteso). Usiamo un piccolo chart di
# testo separato (x/y semplice, non geografico) impilato sopra la mappa con
# vconcat, invece di posizionarlo DENTRO la mappa con longitude/latitude —
# più semplice e robusto.
year_label = alt.Chart(time_base).transform_filter(
    year_select
).transform_aggregate(
    anno='max(year)'
).mark_text(
    fontSize=28, fontWeight='bold', color='#333333', align='center'
).encode(
    text=alt.Text('anno:Q', format='.0f')
).properties(width=750, height=40)

map_only = (background + choropleth_time).properties(
    width=750, height=420,
    title=alt.TitleParams(
        text='Cluster socioeconomico dei paesi olimpici, edizione per edizione',
        subtitle="Trascina lo slider per scorrere le edizioni olimpiche — i colori "
                 "mostrano il cluster (dinamico) del paese in quell'anno",
        fontSize=14, subtitleFontSize=11
    )
)

map_time = alt.vconcat(year_label, map_only).configure_view(stroke=None)

map_time.save('mappa_cluster_temporale.html')
print("✓ Salvato: mappa_cluster_temporale.html")
map_time

In [ ]:
# %% [6] HEATMAP ALTAIR — CHI DOMINA OGNI SPORT?
# -----------------------------------------------------------------------------
heatmap_data = sport_cluster.copy()
sport_totals = heatmap_data.groupby('Sport')['count'].sum().reset_index()
sport_totals.columns = ['Sport', 'sport_total']
heatmap_data = heatmap_data.merge(sport_totals, on='Sport')
heatmap_data['pct_in_sport'] = (
    heatmap_data['count'] / heatmap_data['sport_total'] * 100
).round(1)

top20 = (
    sport_df.groupby('Sport').size()
    .sort_values(ascending=False).head(20).index.tolist()
)
heatmap_data = heatmap_data[heatmap_data['Sport'].isin(top20)]

heatmap = alt.Chart(heatmap_data).mark_rect(stroke='white', strokeWidth=0.5).encode(
    x=alt.X('cluster_label:N',
            sort=CLUSTER_ORDER,
            title='',
            axis=alt.Axis(labelAngle=-30)),
    y=alt.Y('Sport:N',
            sort=alt.EncodingSortField(field='pct_in_sport', op='max',
                                       order='descending'),
            title=''),
    color=alt.Color(
        'pct_in_sport:Q',
        scale=alt.Scale(scheme='blues'),
        legend=alt.Legend(title='% medaglie nello sport')
    ),
    tooltip=[
        alt.Tooltip('Sport:N',           title='Sport'),
        alt.Tooltip('cluster_label:N',   title='Cluster'),
        alt.Tooltip('pct_in_sport:Q',    title='% nello sport', format='.1f'),
        alt.Tooltip('count:Q',           title='N. eventi'),
    ]
).properties(
    width=320, height=500,
    title=alt.TitleParams(
        text='Chi domina ogni sport?',
        subtitle='% medaglie vinte da ogni cluster (per sport)',
        fontSize=14, subtitleFontSize=11
    )
)

text_layer = alt.Chart(heatmap_data).mark_text(fontSize=9).encode(
    x=alt.X('cluster_label:N', sort=CLUSTER_ORDER),
    y=alt.Y('Sport:N', sort=alt.EncodingSortField(
        field='pct_in_sport', op='max', order='descending')),
    text=alt.Text('pct_in_sport:Q', format='.0f'),
    color=alt.condition(
        alt.datum.pct_in_sport > 50,
        alt.value('white'),
        alt.value('black')
    )
)

heatmap_final = (heatmap + text_layer).configure_axis(labelFontSize=10)

heatmap_final.save('heatmap_sport_cluster_altair.html')
print("✓ Salvato: heatmap_sport_cluster_altair.html")
heatmap_final

## MAPPA DEL MONDO — CLUSTER STATICO (media pluriennale)

A differenza della prima mappa (cluster più recente disponibile per paese) e di quella con slider (cluster dell'anno specifico), questa mappa usa il **cluster statico** di `04_Cluster_Statico.ipynb`: un'unica etichetta per paese, calcolata sulla media di tutti gli indicatori su tutte le edizioni 1964–2024. Passando il mouse su un paese, il tooltip mostra **tutti e cinque i valori usati nel K-means** (PIL pro capite, aspettativa di vita, urbanizzazione, mortalità infantile, iscrizione scolastica primaria — medie di lungo periodo), non solo il cluster assegnato.

**File necessario:** `country_avg_static.csv` (prodotto da `04_Cluster_Statico.ipynb`, cella `[S20]`)

In [ ]:
# %% [7] PREPARAZIONE DATI — CLUSTER STATICO
# -----------------------------------------------------------------------------
country_avg_static = pd.read_csv('country_avg_static.csv')

# Stesso mapping IOC -> ISO3 -> numeric_id usato per la mappa del cluster_wb
country_avg_static['iso3'] = country_avg_static['noc_ioc'].map(IOC_TO_ISO3)
mask_missing = country_avg_static['iso3'].isna()
country_avg_static.loc[mask_missing, 'iso3'] = country_avg_static.loc[mask_missing, 'noc_ioc'].apply(
    lambda x: x if x in ISO3_TO_NUMERIC else None
)
country_avg_static['numeric_id'] = country_avg_static['iso3'].map(ISO3_TO_NUMERIC)

map_data_static = country_avg_static[country_avg_static['numeric_id'].notna()].copy()
map_data_static['numeric_id'] = map_data_static['numeric_id'].apply(lambda x: f'{int(x):03d}')

print(f"Paesi mappabili (cluster statico): {len(map_data_static)}")
print(map_data_static['cluster_label_static'].value_counts().reindex(CLUSTER_ORDER).to_string())

In [ ]:
# %% [7bis] MAPPA DEL MONDO — CLUSTER STATICO CON ETICHETTE DEI VALORI K-MEANS
# -----------------------------------------------------------------------------
# Il tooltip (etichetta interattiva al passaggio del mouse) mostra tutti e
# cinque gli indicatori usati per calcolare il cluster statico via K-means,
# non solo l'etichetta finale — così si può verificare "perché" un paese è
# stato assegnato a un determinato cluster.

choropleth_static = alt.Chart(world_geo).mark_geoshape(
    stroke='white', strokeWidth=0.4
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(
        data=map_data_static,
        key='numeric_id',
        fields=['country', 'cluster_label_static', 'gdp_per_capita', 'life_expectancy',
                'urbanization_pct', 'infant_mortality', 'primary_school_enrollment']
    )
).transform_filter(
    'datum.cluster_label_static != null'
).encode(
    color=alt.Color(
        'cluster_label_static:N',
        scale=alt.Scale(domain=CLUSTER_ORDER, range=CLUSTER_COLORS),
        legend=alt.Legend(
            title='Cluster socioeconomico statico (media 1964–2024)',
            orient='bottom',
            titleFontSize=11,
            labelFontSize=10,
            columns=2,
        )
    ),
    tooltip=[
        alt.Tooltip('country:N',                    title='Paese'),
        alt.Tooltip('cluster_label_static:N',        title='Cluster (statico)'),
        alt.Tooltip('gdp_per_capita:Q',              title='PIL pro capite medio (USD)', format=',.0f'),
        alt.Tooltip('life_expectancy:Q',             title='Aspettativa di vita media', format='.1f'),
        alt.Tooltip('urbanization_pct:Q',            title='Urbanizzazione media (%)', format='.1f'),
        alt.Tooltip('infant_mortality:Q',            title='Mortalità infantile media', format='.1f'),
        alt.Tooltip('primary_school_enrollment:Q',   title='Iscriz. scuola primaria media (%)', format='.1f'),
    ]
).project('naturalEarth1')

map_cluster_static = (background + choropleth_static).properties(
    width=750, height=420,
    title=alt.TitleParams(
        text='Cluster socioeconomico statico dei paesi olimpici',
        subtitle=('Media di tutti gli indicatori usati nel K-means (1964–2024) — '
                  'passa il mouse su un paese per vedere i valori'),
        fontSize=14, subtitleFontSize=11
    )
).configure_view(stroke=None)

map_cluster_static.save('mappa_cluster_statico.html')
print("✓ Salvato: mappa_cluster_statico.html")
map_cluster_static